# llm-cost-pilot — Experiment Analysis

Visualize optimization progress from `results.tsv`.

In [ ]:
import csv
import matplotlib.pyplot as plt

# Load results
experiments = []
with open("results.tsv") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        experiments.append(row)

print(f"Loaded {len(experiments)} experiments")
if experiments:
    print(f"Columns: {list(experiments[0].keys())}")

In [ ]:
# Parse numeric fields
ids = list(range(len(experiments)))
cpq = [float(e["cost_per_quality"]) for e in experiments]
savings = [float(e["savings_pct"]) for e in experiments]
quality = [float(e["avg_quality"]) for e in experiments]
status = [e["status"] for e in experiments]

# Color by status
colors = ["green" if s == "kept" else "red" if s == "reverted" else "blue" for s in status]

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Cost per quality over experiments
axes[0].scatter(ids, cpq, c=colors, alpha=0.7, edgecolors="black", linewidth=0.5)
axes[0].set_ylabel("Cost per Quality ($)")
axes[0].set_title("llm-cost-pilot — Optimization Progress")
axes[0].grid(True, alpha=0.3)

# Savings percentage trend
axes[1].bar(ids, savings, color=colors, alpha=0.7, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("Savings (%)")
axes[1].axhline(y=0, color="black", linestyle="-", linewidth=0.5)
axes[1].grid(True, alpha=0.3)

# Quality floor
axes[2].scatter(ids, quality, c=colors, alpha=0.7, edgecolors="black", linewidth=0.5)
axes[2].axhline(y=0.85, color="red", linestyle="--", label="Quality floor (0.85)")
axes[2].set_ylabel("Avg Quality")
axes[2].set_xlabel("Experiment")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("optimization_progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved optimization_progress.png")

In [ ]:
# Summary statistics
kept = [e for e in experiments if e["status"] == "kept"]
reverted = [e for e in experiments if e["status"] == "reverted"]

print(f"Total experiments: {len(experiments)}")
print(f"  Kept:     {len(kept)}")
print(f"  Reverted: {len(reverted)}")

if kept:
    best = min(kept, key=lambda e: float(e["cost_per_quality"]))
    print(f"\nBest experiment:")
    print(f"  ID:          {best['experiment_id']}")
    print(f"  Description: {best['description']}")
    print(f"  Cost/Quality: ${float(best['cost_per_quality']):.4f}")
    print(f"  Savings:      {best['savings_pct']}%")
    print(f"  Quality:      {best['avg_quality']}")